In [1]:
import sounddevice as sd
from scipy.io.wavfile import write
import numpy as np
import parselmouth
from tensorflow.keras.models import load_model
from sincnet_tensorflow import SincConv1D, LayerNorm
import os

Normalizar audios para input de la red neuronal

In [ ]:
import os
import numpy as np
import parselmouth

def preprocess_and_save_audio(file_path, max_length, output_dir):
    try:
        # Cargar el archivo de audio
        sound = parselmouth.Sound(file_path)
        
        # Obtener la tasa de muestreo
        sample_rate = sound.sampling_frequency
        
        # Obtener el número de canales
        num_channels = sound.n_channels
        
        # Convertir el audio a un array de numpy
        audio_array = sound.values
        
        # Verificar el contenido del array de audio
        if audio_array.size == 0:
            print(f"El archivo '{file_path}' está vacío.")
            return
        
        # Si el audio tiene 2 canales, tomar solo el primer canal para simplificar
        if num_channels == 2:
            # Asegurarse de tomar el canal correcto dependiendo de la forma del array
            if audio_array.shape[0] == 2:  # Formato (2, N)
                audio_array = audio_array[0, :]  # Tomar el primer canal (canal izquierdo)
            elif audio_array.shape[1] == 2:  # Formato (N, 2)
                audio_array = audio_array[:, 0]  # Tomar el primer canal (canal izquierdo)
        
        # Calcular el número de muestras
        num_samples = len(audio_array)
        
        # Ajustar la longitud del audio
        if num_samples > max_length:
            audio_array = audio_array[:max_length]
        else:
            audio_array = np.pad(audio_array, (0, max_length - num_samples), mode='constant')
        
        # Crear un nuevo objeto Sound con el array de audio ajustado
        new_sound = parselmouth.Sound(audio_array, sampling_frequency=sample_rate)
        
        # Verificar las propiedades del nuevo objeto Sound
        print(f"Duración del nuevo sonido: {new_sound.duration} segundos")
        print(f"Número de canales del nuevo sonido: {new_sound.n_channels}")
        print(f"Forma del array de valores del nuevo sonido: {new_sound.values.shape}")
        
        # Guardar el archivo de audio procesado en la carpeta de salida
        output_file_path = os.path.join(output_dir, os.path.basename(file_path))
        new_sound.save(output_file_path, format="WAV")
        
        print(f"Archivo procesado y guardado en: {output_file_path}")
    except Exception as e:
        print(f"Error al procesar el archivo '{file_path}': {e}")

# Ruta de los archivos de audio
audio_dir = 'AudiosTest'
output_dir = 'PaddedTest'

# Crear la carpeta de salida si no existe
os.makedirs(output_dir, exist_ok=True)

# Define la longitud máxima en muestras
max_length = int(18.058979166666667 * 48000)  # Frecuencia de muestreo de 48,000 Hz

# Preprocesar y guardar todos los audios
for file in os.listdir(audio_dir):
    if file.endswith('.wav'):
        file_path = os.path.join(audio_dir, file)
        preprocess_and_save_audio(file_path, max_length, output_dir)

Prueba del modelo

In [2]:
class SincConv1DWithConfig(SincConv1D):
    def get_config(self):
        # Llama al método get_config de la superclase
        config = super(SincConv1DWithConfig, self).get_config()
        
        # Agrega los parámetros específicos de SincConv1D
        config.update({
            "N_filt": self.N_filt,
            "Filt_dim": self.Filt_dim,
            "fs": self.fs,
            "stride": self.stride,
            "padding": self.padding,
        })
        return config
    
class LayerNormWithConfig(LayerNorm):
    def get_config(self):
        config = super(LayerNormWithConfig, self).get_config()
    
        return config


In [3]:
    # Cargar el modelo guardado
model = load_model('ProsodyNetwork.h5', custom_objects={
    'SincConv1DWithConfig': SincConv1DWithConfig,
    'LayerNormWithConfig': LayerNormWithConfig
})

In [7]:
print(model.input_shape)


(None, 866831, 1)


In [4]:
import os
import numpy as np
import pandas as pd
import parselmouth
import tensorflow as tf

# Función para cargar audio usando Parselmouth
def load_audio(file_path):
    """ Cargar un archivo de audio y devolver un array de numpy """
    sound = parselmouth.Sound(file_path)
    audio_array = sound.values.flatten()  # Aplanar el array a una dimensión
    return audio_array

# Cargar el CSV con las etiquetas
df = pd.read_csv('test.csv')

# Obtener las etiquetas de las columnas 4 a 12 (1-9)
y_test = df.iloc[:, 3:12].values  # Columnas del 1 al 9 (empezando desde la cuarta columna)

# Ruta de la carpeta de audios de test
audio_folder = 'PaddedTest'

# Procesar los audios de la carpeta
X_test = []
for audio_name in df['nombre']:  # Suponiendo que la columna 'nombre' contiene los nombres de los archivos
    file_path = os.path.join(audio_folder, audio_name)
    if os.path.exists(file_path):
        audio_array = load_audio(file_path)
        X_test.append(audio_array)
    else:
        print(f"Archivo no encontrado: {file_path}")

# Convertir la lista a un array de NumPy
X_test = np.array(X_test, dtype=np.float32)

# Asegurar que los datos tienen la forma esperada (batch_size, timesteps, features)
X_test = X_test[..., np.newaxis]  # Añadir dimensión de canal para la CNN

# Realizar la predicción con el modelo cargado
predictions = model.predict(X_test)

# Evaluar el modelo comparando con las etiquetas reales
loss, accuracy = model.evaluate(X_test, y_test, verbose=1)

print(f"Pérdida en test: {loss:.4f}")
print(f"Precisión en test: {accuracy:.4f}")


8/8 [==============================] - 14s 1s/step - loss: 0.6053 - binary_accuracy: 0.6838
Pérdida en test: 0.6053
Precisión en test: 0.6838


In [5]:
from sklearn.metrics import f1_score

# Supongamos que y_true y y_pred son tus etiquetas verdaderas y las predicciones del modelo
predictions = model.predict(X_test)
predictions_binary = (predictions > 0.4).astype(int)  # Convertir las probabilidades en etiquetas binarias

f1_scores = f1_score(y_test, predictions_binary, average=None)  # F1 Score para cada etiqueta
average_f1_score = f1_score(y_test, predictions_binary, average='macro')  # F1 Score promedio macro

print("F1 Score por etiqueta:", f1_scores)
print("F1 Score promedio (macro):", average_f1_score)

8/8 [==============================] - 12s 1s/step
F1 Score por etiqueta: [0.51497006 0.         0.56221198 0.41767068 0.48587571 0.38333333
 0.36585366 0.         0.26415094]
F1 Score promedio (macro): 0.33267404062875755


In [6]:
import numpy as np

# Ejemplo de umbrales específicos para cada etiqueta
thresholds = [0.7, 0.5, 0.6, 0.45, 0.5, 0.7, 0.4, 0.7, 0.45]

# Suponiendo que y_pred tiene las probabilidades predichas (dimensión: [num_samples, 9])
predictions_adjusted = np.zeros_like(predictions)

for i in range(predictions.shape[1]):
    predictions_adjusted[:, i] = (predictions[:, i] >= thresholds[i]).astype(int)

# y_pred_adjusted ahora tiene las predicciones finales con los umbrales ajustados


In [7]:
from sklearn.metrics import f1_score

# Calcula el F1 Score para cada etiqueta y el promedio
f1_scores = f1_score(y_test, predictions_adjusted, average=None)
f1_macro = f1_score(y_test, predictions_adjusted, average='macro')
f1_micro = f1_score(y_test, predictions_adjusted, average='micro')

print("F1 Score para cada etiqueta:", f1_scores)
print("F1 Score macro promedio:", f1_macro)
print("F1 Score micro promedio:", f1_micro)

F1 Score para cada etiqueta: [0.11764706 0.         0.46052632 0.39461883 0.46753247 0.44303797
 0.36585366 0.         0.28571429]
F1 Score macro promedio: 0.2816589550178448
F1 Score micro promedio: 0.3806763285024155


MFCC SVM

In [23]:
import os
import numpy as np
import pandas as pd
import librosa
from scipy.fft import dct
import joblib

# Función para extraer características MFCC (DCT sobre Mel-espectrograma)
def extract_features(audio_path, sample_rate=22050, n_mels=13, n_dct=13):
    y, sr = librosa.load(audio_path, sr=sample_rate)
    
    # Calcular el Mel-espectrograma
    mel_spectrogram = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels)
    
    # Convertir el espectrograma a escala logarítmica
    mel_db = librosa.power_to_db(mel_spectrogram, ref=np.max)
    
    # Aplicar DCT sobre el espectrograma
    mfcc_features = dct(mel_db, axis=0, norm='ortho')[:n_dct]  # Selección de los primeros n_dct coeficientes
    return mfcc_features.flatten()  # Aplanar para usar como entrada del modelo

# Cargar el CSV con las etiquetas
df = pd.read_csv('test.csv')

# Obtener las etiquetas de las columnas 4 a 12 (1-9)
y_test = df.iloc[:, 3:12].values  # Columnas del 1 al 9 (empezando desde la cuarta columna)

# Ruta de la carpeta de audios de test
audio_folder = 'PaddedTest'

# Cargar el modelo SVM entrenado
svm_model = joblib.load('svm_model.joblib')

# Procesar los audios de la carpeta y extraer las características MFCC
X_test = []
for audio_name in df['nombre']:  # Los nombres ya están ordenados en el CSV
    file_path = os.path.join(audio_folder, audio_name)
    if os.path.exists(file_path):
        audio_features = extract_features(file_path)
        X_test.append(audio_features)
    else:
        print(f"Archivo no encontrado: {file_path}")

# Convertir la lista a un array de NumPy
X_test = np.array(X_test, dtype=np.float32)

# Realizar la predicción con el modelo SVM cargado
predictions = svm_model.predict(X_test)

# Evaluar el modelo comparando con las etiquetas reales
accuracy_per_label = []
for i in range(y_test.shape[1]):
    # Comparar las predicciones por etiqueta
    acc = np.mean(predictions[:, i] == y_test[:, i])
    accuracy_per_label.append(acc)
    print(f"Etiqueta {i + 1}: Accuracy = {acc:.4f}")

# Métricas generales
mean_accuracy = np.mean(accuracy_per_label)
print(f"\nMean Accuracy (promedio por etiqueta): {mean_accuracy:.4f}")


Etiqueta 1: Accuracy = 0.2532
Etiqueta 2: Accuracy = 0.0129
Etiqueta 3: Accuracy = 0.3734
Etiqueta 4: Accuracy = 0.2618
Etiqueta 5: Accuracy = 0.2704
Etiqueta 6: Accuracy = 0.2275
Etiqueta 7: Accuracy = 0.2704
Etiqueta 8: Accuracy = 0.0944
Etiqueta 9: Accuracy = 0.1416

Mean Accuracy (promedio por etiqueta): 0.2117


STFT con CNN

Crear imágenes stft con datos de test

In [24]:
import os
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt

# Crear la carpeta "audio_frecuencia_not_padded" si no existe
output_dir = 'STFT'
os.makedirs(output_dir, exist_ok=True)

# Ruta de la carpeta con los audios de test
audio_folder = 'PaddedTest'

# Procesar los archivos de audio en la carpeta
for file_name in os.listdir(audio_folder):
    if file_name.endswith('.wav'):  # Solo procesar archivos .wav
        file_path = os.path.join(audio_folder, file_name)
        
        # Cargar el archivo de audio
        y_audio, sr = librosa.load(file_path)

        # Calcular el espectrograma del audio completo
        D = librosa.amplitude_to_db(np.abs(librosa.stft(y_audio)), ref=np.max)

        # Crear el gráfico del espectrograma en color
        plt.figure(figsize=(12, 8))
        librosa.display.specshow(D, sr=sr, x_axis='time', y_axis='log', cmap='viridis')  # Usar 'viridis' para colores
        plt.colorbar(format='%+2.0f dB')
        plt.title(f'Espectrograma de amplitud del audio completo: {file_name}')
        plt.xlabel('Tiempo (s)')
        plt.ylabel('Frecuencia (Hz)')
        
        # Guardar el gráfico en la subcarpeta correspondiente con el mismo nombre que el archivo
        output_file = os.path.join(output_dir, f'{file_name}.png')
        plt.savefig(output_file)
        plt.close()  # Cerrar la figura para liberar memoria


Generar X

In [25]:
import os
import numpy as np
import pandas as pd
from PIL import Image

def load_images_from_folder(image_folder, csv_file, input_size=(224, 224)):
    # Cargar el CSV con las rutas de los nombres de archivo
    df = pd.read_csv(csv_file)

    # Lista para almacenar las imágenes cargadas
    X = []

    # Recorrer la columna 'nombre' del CSV
    for audio_name in df['nombre']:  # Asegúrate de que 'nombre' es la columna correcta en tu CSV
        # Crear el nombre del archivo de la imagen correspondiente
        image_name = f"{audio_name}.png"
        image_path = os.path.join(image_folder, image_name)

        # Comprobar si el archivo existe
        if os.path.isfile(image_path):
            try:
                # Cargar la imagen generada del espectrograma
                img = Image.open(image_path)

                # Convertir la imagen a RGB (eliminando cualquier canal alfa)
                img_rgb = img.convert("RGB")

                # Redimensionar la imagen al tamaño especificado
                img_resized = img_rgb.resize(input_size)

                # Convertir la imagen a una matriz numpy
                img_array_resized = np.array(img_resized)

                # Agregar la imagen a la lista
                X.append(img_array_resized)

            except Exception as e:
                print(f"Error al procesar la imagen {image_name}: {e}")

    # Convertir la lista en un array numpy
    X = np.array(X)

    # Devolver el array de imágenes
    return X


In [26]:
image_folder = 'STFT'
X = load_images_from_folder(image_folder, 'test.csv')

# Verificar la forma final de las imágenes cargadas
print(f"Imágenes cargadas en la variable X con forma: {X.shape}")

Imágenes cargadas en la variable X con forma: (233, 224, 224, 3)


Generar y

In [28]:
# Cargar el CSV con las etiquetas
df = pd.read_csv('test.csv')

# Obtener las etiquetas de las columnas 4 a 12 (1-9)
y_test = df.iloc[:, 3:12].values  # Columnas del 1 al 9 (empezando desde la cuarta columna)

print(f"La variable y con forma: {y_test.shape}")

La variable y con forma: (233, 9)


Probar modelo con datos de test

In [29]:
from tensorflow.keras.models import load_model

# Cargar el modelo previamente guardado
model = load_model('stft.h5')

# Si deseas evaluar el modelo:
loss, accuracy = model.evaluate(X, y_test)
print(f'Pérdida: {loss:.4f}, Precisión: {accuracy:.4f}')


8/8 [==============================] - 3s 303ms/step - loss: 0.5538 - binary_accuracy: 0.7678
Pérdida: 0.5538, Precisión: 0.7678


MEL SPECTOGRAM con CNN

Generar Mel para variable X

In [31]:
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import os

# Ruta de entrada y salida
input_folder = 'AudiosTest'  # Carpeta con los archivos .wav
output_folder = 'mel_spectrograms_vggish'  # Carpeta para guardar las imágenes
os.makedirs(output_folder, exist_ok=True)

# Parámetros para el espectrograma
target_sr = 16000  # Frecuencia de muestreo
n_mels = 64
n_fft = 400
hop_length = 160

# Procesar los archivos de audio
for file_name in os.listdir(input_folder):
    if file_name.endswith('.wav'):  # Procesar solo archivos .wav
        file_path = os.path.join(input_folder, file_name)
        try:
            # Cargar el archivo de audio
            y, sr = librosa.load(file_path, sr=target_sr)

            # Crear el espectrograma de Mel
            mel_spectrogram = librosa.feature.melspectrogram(
                y=y,
                sr=sr,
                n_fft=n_fft,
                hop_length=hop_length,
                n_mels=n_mels,
                fmin=125,
                fmax=7500
            )

            # Convertir a escala logarítmica
            log_mel_spectrogram = librosa.power_to_db(mel_spectrogram, ref=np.max)

            # Asegurar que tenga 96 frames temporales
            if log_mel_spectrogram.shape[1] < 96:
                pad_width = 96 - log_mel_spectrogram.shape[1]
                log_mel_spectrogram = np.pad(log_mel_spectrogram, ((0, 0), (0, pad_width)), mode='constant')
            else:
                log_mel_spectrogram = log_mel_spectrogram[:, :96]

            # Crear una figura sin ejes ni colorbar
            plt.figure(figsize=(2.56, 1.71))  # Tamaño inicial aproximado en pulgadas
            plt.axis('off')  # Ocultar los ejes
            librosa.display.specshow(log_mel_spectrogram, sr=sr, hop_length=hop_length, fmin=125, fmax=7500, cmap='viridis')

            # Guardar la imagen temporalmente
            temp_output_path = os.path.join(output_folder, f"temp_{os.path.splitext(file_name)[0]}.png")
            plt.savefig(temp_output_path, dpi=96, bbox_inches='tight', pad_inches=0)
            plt.close()

            # Cargar la imagen temporal y redimensionar a 96x64
            img = Image.open(temp_output_path)
            img_resized = img.resize((96, 64), Image.Resampling.LANCZOS)

            img_array = np.array(img_resized)

            if img_array.shape[-1] == 4:  # Si la imagen tiene 4 canales (viridis incluye una dimension alfa)
                img_array = img_array[:, :, :3]  # Mantener solo los tres primeros canales (RGB)

            # Convertir nuevamente a imagen PIL
            img_resized_rgb = Image.fromarray(img_array)

            # Guardar la imagen final
            final_output_path = os.path.join(output_folder, f"{os.path.splitext(file_name)[0]}.png")
            img_resized_rgb.save(final_output_path)

            # Eliminar la imagen temporal
            os.remove(temp_output_path)

        except Exception as e:
            print(f"Error procesando {file_name}: {e}")

In [34]:
# Ruta a la carpeta de imágenes
image_folder = 'mel_spectrograms_vggish'
# Lista para almacenar las imágenes cargadas
X = []
# Cargar el CSV con las etiquetas
df = pd.read_csv('test.csv')
# Recorrer las posibles combinaciones de x y y
for audio_name in df['nombre']:  # Asegúrate de que 'nombre' es la columna correcta en tu CSV
        # Crear el nombre del archivo de la imagen correspondiente
        base_name = os.path.splitext(audio_name)[0]  # Quita la extensión .wav
        image_name = f"{base_name}.png"
        image_path = os.path.join(image_folder, image_name)

        # Comprobar si el archivo existe
        if os.path.isfile(image_path):
            try:
                # Cargar la imagen generada del espectrograma
                img = Image.open(image_path)

                # Convertir la imagen a una matriz numpy
                img_array_resized = np.array(img)

                # Agregar la imagen a la lista
                X.append(img_array_resized)

            except Exception as e:
                print(f"Error al procesar la imagen {image_name}: {e}")

# Convertir la lista en un array numpy
X = np.array(X)

# Verificar la forma final de las imágenes cargadas
print(f"Imágenes cargadas en la variable X con forma: {X.shape}")

Imágenes cargadas en la variable X con forma: (233, 64, 96, 3)


Generar y

In [35]:
# Cargar el CSV con las etiquetas
df = pd.read_csv('test.csv')

# Obtener las etiquetas de las columnas 4 a 12 (1-9)
y_test = df.iloc[:, 3:12].values  # Columnas del 1 al 9 (empezando desde la cuarta columna)

print(f"La variable y con forma: {y_test.shape}")

La variable y con forma: (233, 9)


Probar modelo

In [37]:
from tensorflow.keras.models import load_model

# Cargar el modelo previamente guardado
model = load_model('mel.h5')

# Si deseas evaluar el modelo:
loss, accuracy = model.evaluate(X, y_test)
print(f'Pérdida: {loss:.4f}, Precisión: {accuracy:.4f}')

8/8 [==============================] - 0s 39ms/step - loss: 0.5761 - binary_accuracy: 0.6938
Pérdida: 0.5761, Precisión: 0.6938
